### PIMA dataset

In [61]:
from scipy import stats
import pandas as pd
import numpy as np

!gdown --id '17BhmQ08NEtvn7WwPp-OcZwz-2u-cceZm'  --output data.csv
data = pd.read_csv('/content/data.csv', encoding = 'latin1')
data.head()

/usr/local/lib/python3.10/dist-packages/gdown/cli.py:121: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=17BhmQ08NEtvn7WwPp-OcZwz-2u-cceZm
To: /content/data.csv
100% 11.4k/11.4k [00:00<00:00, 26.1MB/s]


,Glucose,BloodPressure,Insulin,Age,Outcome
0,89,66,94,21,0
1,137,40,168,33,2
2,78,50,88,26,0
3,197,70,543,53,2
4,189,60,846,59,2


In [62]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from xgboost import plot_importance
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
import keras
from keras.utils import to_categorical

X = data.drop('Outcome', axis=1)
Y = data['Outcome']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, shuffle = True, test_size = 0.3, random_state = 87)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Y_train shape:", Y_train.shape)
print("Y_test shape:", Y_test.shape)

X_train shape: (509, 4)
X_test shape: (219, 4)
Y_train shape: (509,)
Y_test shape: (219,)


In [63]:
class Model1XGboost:
  def __init__(self):
    self.model = None
    self.scaler = None

  def fit(self, X_train, X_test, Y_train, Y_test):
    self.model = XGBClassifier(max_depth=10, learning_rate=0.1, n_estimators=1000,
                      reg_alpha=0.005, subsample=0.8,
                      gamma=0, objective='binary:logistic')
    self.scaler = StandardScaler()
    columns = X_train.columns
    indexs_train = X_train.index
    X_train = pd.DataFrame(self.scaler.fit_transform(X_train), index=indexs_train, columns=columns)
    indexs_test = X_test.index
    X_test = pd.DataFrame(self.scaler.transform(X_test), index=indexs_test, columns=columns)
    self.model.fit(X_train, Y_train)

  def predict(self, X_test):
    Y_pred = self.model.predict(X_test)
    return Y_pred

In [64]:
from sklearn.datasets import make_blobs
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import seaborn as sns
from keras.utils import to_categorical

class Model1KNN:
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()

    def fit(self, X_train, X_test, Y_train, Y_test):
        scaler = StandardScaler()
        X_train = self.scaler.fit_transform(X_train)
        X_test = self.scaler.transform(X_test)

        k_values = [i for i in range(1, 31)]
        scores = []

        for k in k_values:
            knn = KNeighborsClassifier(n_neighbors=k)
            score = cross_val_score(knn, X_train, Y_train, cv=5)
            scores.append(np.mean(score))

        best_index = np.argmax(scores)
        best_k = k_values[best_index]

        self.model = KNeighborsClassifier(n_neighbors=best_k)
        self.model.fit(X_train, Y_train)

    def predict(self, X_test):
        Y_pred = self.model.predict(X_test)
        return Y_pred

In [65]:
model1_xgboost = Model1XGboost()
model1_knn = Model1KNN()


# model fitting
model1_xgboost.fit(X_train, X_test, Y_train, Y_test)
model1_knn.fit(X_train, X_test, Y_train, Y_test)

# Base model
base_models = [('xgboost', model1_xgboost.model), ('knn', model1_knn.model)]

# Logistic Regression as meta model
meta_model = LogisticRegression()

# StackingClassifier
stacking_classifier = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)

stacking_classifier.fit(X_train, Y_train)

stacking_predictions = stacking_classifier.predict(X_test)

print("Stacking Classifier Classification Report:")
print(classification_report(Y_test, stacking_predictions, digits = 8))

print("Accuracy Score:", accuracy_score(Y_test, stacking_predictions))
print("Precision Score:", precision_score(Y_test, stacking_predictions, average='weighted'))
print("Recall Score:", recall_score(Y_test, stacking_predictions, average='weighted'))
print("F1 Score:", f1_score(Y_test, stacking_predictions, average='weighted'))

Stacking Classifier Classification Report:
              precision    recall  f1-score   support

           0  1.00000000 1.00000000 1.00000000        54
           1  1.00000000 1.00000000 1.00000000        79
           2  1.00000000 1.00000000 1.00000000        86

    accuracy                      1.00000000       219
   macro avg  1.00000000 1.00000000 1.00000000       219
weighted avg  1.00000000 1.00000000 1.00000000       219

Accuracy Score: 1.0
Precision Score: 1.0
Recall Score: 1.0
F1 Score: 1.0


### LMCH dataset

In [66]:
!gdown --id '1-0NT8HBlFAkB_lMSxE8C0fMiy5V__0Zt'  --output data.csv
data = pd.read_csv('/content/data.csv', encoding = 'latin1')
data.head()


/usr/local/lib/python3.10/dist-packages/gdown/cli.py:121: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1-0NT8HBlFAkB_lMSxE8C0fMiy5V__0Zt
To: /content/data.csv
100% 25.4k/25.4k [00:00<00:00, 42.0MB/s]


,AGE,HbA1c,Chol,TG,HDL,VLDL,Output
0,50,4.9,4.2,0.9,2.4,0.5,0
1,26,4.9,3.7,1.4,1.1,0.6,0
2,50,4.9,4.2,0.9,2.4,0.5,0
3,50,4.9,4.2,0.9,2.4,0.5,0
4,33,4.9,4.9,1.0,0.8,0.4,0


In [67]:
from sklearn.preprocessing import LabelEncoder
import keras
from keras.utils import to_categorical
#Label Encoding
labels = data['Output']
# encode class values as integers
encoder = LabelEncoder()
encoder.fit(labels)
encoded_Y = encoder.transform(labels)
# convert integers to dummy variables (i.e. one hot encoded)
dummy_y = to_categorical(encoded_Y)
# Remove the labels from the features
# axis 1 refers to the columns
data = data.drop('Output', axis = 1)

# Saving feature names for later use
data_list = list(data.columns)

X_train, X_test, Y_train, Y_test = train_test_split(data, dummy_y, shuffle = True, test_size = 0.3, random_state = 87)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Y_train shape:", Y_train.shape)
print("Y_test shape:", Y_test.shape)

X_train shape: (699, 6)
X_test shape: (300, 6)
Y_train shape: (699, 5)
Y_test shape: (300, 5)


In [70]:
model1_xgboost = Model1XGboost()
model1_knn = Model1KNN()

# model fitting
model1_xgboost.fit(X_train, X_test, Y_train, Y_test)
model1_knn.fit(X_train, X_test, Y_train, Y_test)

# Base model
base_models = [('xgboost', model1_xgboost.model), ('knn', model1_knn.model)]

# Logistic Regression as meta model
meta_model = LogisticRegression()

# StackingClassifier
stacking_classifier = StackingClassifier(estimators=base_models, final_estimator=meta_model, cv=5)

stacking_classifier.fit(X_train, np.argmax(Y_train, axis=1))

stacking_predictions = stacking_classifier.predict(X_test)

print("Stacking Classifier Classification Report:")
print(classification_report(np.argmax(Y_test, axis=1), stacking_predictions, digits=8))


print("Accuracy Score:", accuracy_score(np.argmax(Y_test, axis=1), stacking_predictions))
print("Precision Score:", precision_score(np.argmax(Y_test, axis=1), stacking_predictions, average='weighted'))
print("Recall Score:", recall_score(np.argmax(Y_test, axis=1), stacking_predictions, average='weighted'))
print("F1 Score:", f1_score(np.argmax(Y_test, axis=1), stacking_predictions, average='weighted'))

/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


Stacking Classifier Classification Report:
              precision    recall  f1-score   support

           0  0.90000000 0.84375000 0.87096774        32
           1  1.00000000 0.93750000 0.96774194        16
           2  0.96862745 0.99196787 0.98015873       249
           3  0.00000000 0.00000000 0.00000000         1
           4  0.00000000 0.00000000 0.00000000         2

    accuracy                      0.96333333       300
   macro avg  0.57372549 0.55464357 0.56377368       300
weighted avg  0.95329412 0.96333333 0.95804788       300

Accuracy Score: 0.9633333333333334
Precision Score: 0.9532941176470588
Recall Score: 0.9633333333333334
F1 Score: 0.9580478750640041


/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_split.py:700: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no pre